# 01 — Prepare the shared 5,000-image dataset

Run once before training. One fixed manifest is used by all models. The noisy input is generated reproducibly from each clean RGB image.

## Step 1 — Locate the project directory. Select the `.venv` kernel in VS Code before running.

In [ ]:
from pathlib import Path
import sys, os
ROOT = Path.cwd()
if not (ROOT / 'app').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'app').is_dir():
    raise FileNotFoundError('Open the complete iMAGE_Notebook_Edition folder in VS Code first.')
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('Project:', ROOT)


## Step 2 — Download the public COCO 2017 validation images automatically (about 1 GB).

This is an optional convenience, not the only possible dataset. If you already have **5,000 distinct clean images**, put them inside `data/raw/val2017/` and skip this cell. Verify dataset license and suitability for your study. Internet is required for this step.

In [ ]:
from app.online import download
from zipfile import ZipFile
archive = ROOT/'data/raw/val2017.zip'
image_dir = ROOT/'data/raw/val2017'
DOWNLOAD_COCO = False  # Change to True to fetch the ~1 GB public archive automatically.
if DOWNLOAD_COCO:
    download('http://images.cocodataset.org/zips/val2017.zip', archive, min_bytes=500_000_000)
    if not image_dir.exists():
        with ZipFile(archive) as z:
            bad = [i.filename for i in z.infolist() if not i.filename.startswith('val2017/') or '..' in Path(i.filename).parts]
            if bad: raise RuntimeError('Unexpected archive paths')
            z.extractall(ROOT/'data/raw')
    print('Images available at:', image_dir)
else:
    print('Download skipped. Set DOWNLOAD_COCO=True or supply your own images to',image_dir)


## Step 3 — Generate and freeze the shared 4,000 / 500 / 500 split

In [ ]:
from scripts.prepare_dataset import prepare
from app.experiments import dataset_counts
image_dir = ROOT/'data/raw/val2017'
manifest = ROOT/'data/splits.csv'
if manifest.exists():
    print('Using existing manifest. Do not regenerate between model runs.')
else:
    prepare(image_dir, manifest, count=5000, seed=42)
print(dataset_counts(manifest))


## Step 4 — Preview matched clean and noisy samples

In [ ]:
from app.data import DenoiseDataset
import matplotlib.pyplot as plt
import numpy as np
samples = DenoiseDataset(ROOT/'data/splits.csv', 'val', patch=256, sigma=25, seed=42)
noisy, clean = samples[0]
fig, axes = plt.subplots(1,2,figsize=(10,4))
for ax, im, title in zip(axes,[clean,noisy],['Clean reference','Noisy input (σ=25/255)']):
    ax.imshow(np.clip(im.permute(1,2,0).numpy(),0,1));ax.set_title(title);ax.axis('off')
plt.show()


**Dataset governance:** 5,000 samples means 5,000 unique source images; keep source images and their derived noisy versions in the same split. Do not use test images for hyperparameter selection. Additional test noise levels are experiments on the same underlying 500-image test partition, not additional samples.